In [1]:
import numpy as np
import torch

np_arr = np.array([1,2,3])
tensor_from_np = torch.from_numpy(np_arr)
tensor_from_np

tensor([1, 2, 3])

In [2]:
np_from_tensor = tensor_from_np.numpy()
np_from_tensor

array([1, 2, 3])

In [3]:
np_arr[0] = 100
tensor_from_np

tensor([100,   2,   3])

In [7]:
if torch.xpu.is_available():
    xpu_tensor = torch.tensor([1,2,3], device='xpu')
    np_from_xpu = xpu_tensor.cpu().numpy()
    print(f'GPU 텐서 -> NumPy 배열: {np_from_xpu}')
else:
    print('XPU 미작동')

GPU 텐서 -> NumPy 배열: [1 2 3]


In [9]:
if torch.xpu.is_available():
    print("xpu 사용 가능!")
else:
    print("cpu 모드로만 진행")

xpu 사용 가능!


In [10]:
device = torch.device("xpu" if torch.xpu.is_available() else "cpu")
print(f"사용할 디바이스: {device}")

사용할 디바이스: xpu


In [11]:
x = torch.tensor([1.0, 2.0, 3.0]).to(device)
print(f"x가 위치한 디바이스: {x.device}")

x가 위치한 디바이스: xpu:0


In [13]:
x_cpu = x.cpu()
print(f"x가 위치한 디바이스: {x.device}")
print(f'x_cpu가 위치한 디바이스: {x_cpu.device}')

x가 위치한 디바이스: xpu:0
x_cpu가 위치한 디바이스: cpu


In [22]:
import time
size = 20000

a_cpu = torch.rand((size, size))
b_cpu = torch.rand((size, size))

start_time = time.time()
_ = torch.mm(a_cpu, b_cpu)
cpu_time = time.time() - start_time
print(f'CPU matmul time: {cpu_time}')

if torch.xpu.is_available():
    a_xpu = a_cpu.to(device)
    b_xpu = b_cpu.to(device)
    torch.xpu.synchronize()
    start_time = time.time()
    _ = torch.mm(a_xpu, b_xpu)
    torch.xpu.synchronize()
    xpu_time = time.time() - start_time
    print(f'XPU matmul time: {xpu_time}')

CPU matmul time: 35.693674087524414
XPU matmul time: 6.0648345947265625


In [23]:
import gc
for name in ["a_cpu", "b_cpu", "a_xpu", "b_xpu"]:
    if name in globals():
        del globals()[name]

gc.collect()
if torch.xpu.is_available():
    torch.xpu.synchronize()
    torch.xpu.memory.empty_cache()